# Aufgabe 2 · Aktenansicht und belegte Lesernotizen

[Aufgabenstellung](README.md) · [Walkthrough](WALKTHROUGH.md) · [Aktenleseführer](../../docs/AKTENLESEFUEHRER.md)

Die Redaktion stösst in einer Zeugenaussage auf eindrucksvolle Abbildungen. Sind das Aufnahmen des Ereignisses? Du erschliesst die Akte als Dokument, hältst Deine Prüfung mit Seitenbeleg fest und untersuchst unterschiedlich gefüllte Quellmetadaten.

**Teil A und B bilden die Aufgabe. Teil C ist eine Vertiefung.**
Ergänze die TODO-Zellen und dokumentiere Deine Entscheidungen. Nicht bearbeitete Abschnitte melden OFFEN.

## 0. Eigenständiger Einstieg

Verwende den Kernel **Python (rothstein-storage-workshop-2026)** und den bereits eingerichteten MongoDB-Server: [Docker/Codespaces](../../README_technical_preparation.md) oder [lokaler Community Server](../../docs/setup/MONGODB_LOKAL.md). Die SQLite-Aufgabe und deren Ergebnisdateien werden nicht benötigt.

Die Eingabe enthält alle 375 Katalogeinträge unter `data/input/document/catalog_documents.jsonl`. Die Originale und die vorbereiteten Eingaben bleiben unverändert.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import pandas as pd
from IPython.display import display
from pprint import pprint

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/mongodb_workshop.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Öffne das Notebook innerhalb des Repositories.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from mongodb_workshop import (prepare_documents, collection_names, mongo_workspace,
                              import_snapshot, validate_note, save_note, probe_note_schema)
from storage_runtime import connection_summary
pd.set_option("display.max_colwidth", 90)
print("MongoDB-Ziel:", connection_summary()["MongoDB"])

DATASET = "uap"
IS_SOLUTION = False
names = collection_names(DATASET, solution=IS_SOLUTION)
inputs = prepare_documents(DATASET)
SOURCE_KEY = "DOW-UAP-D080"
source = next(d for d in inputs["catalog"] if d["source_key"] == SOURCE_KEY)
completed = set()
print("Collections:", names)
print("Katalogeinträge:", len(inputs["catalog"]))
pprint({k:source[k] for k in ["entry_id","source_key","agency","assets","incident_date_raw"]})

## A1 · Eine Akte als Dokument lesen

Der vorbereitete Import übernimmt die technische `entry_id` als `_id`. Aktenkürzel wie `FBI-UAP-D014` bleiben nicht eindeutig. Stellenangaben, Asset-Metadaten und originale Zusatzfelder sind eingebettet; auf andere Einträge verweist `related_entries` mit technischen IDs.

Der Import ersetzt Dokumente unter stabilen IDs und entfernt erst danach überholte IDs aus **der gewählten Katalog-Collection**. Er ist insgesamt nicht atomar. Nach einer Unterbrechung erneut ausführen und auf `IMPORT OK` achten. Die separate Collection `reading_notes` wird vom Import nicht verändert.

In [ ]:
with mongo_workspace(DATASET, IS_SOLUTION) as (db,names):
    first = import_snapshot(db,names,DATASET,inputs)
with mongo_workspace(DATASET, IS_SOLUTION) as (db,names):
    assert db[names["catalog"]].count_documents({}) == 375
    second = import_snapshot(db,names,DATASET,inputs)
    card = db[names["catalog"]].find_one({"_id":source["entry_id"]},
        {"_id":1,"source_key":1,"title":1,"agency":1,"assets":1,"source_metadata":1,"related_entries":1})
assert first == second == {"catalog":375}
completed.add("Import")
print("IMPORT OK")
pprint(card)

### Deine Modellentscheidung

Warum werden die Asset-Metadaten eingebettet, die Original-PDFs aber als Dateien referenziert? Warum kopieren wir die vollständigen Zielakten aus `related_entries` nicht rekursiv in jedes Dokument? Welche Folgen hat die eingebettete Stellenbezeichnung bei einer nachträglichen Namenskorrektur?

**Deine Begründung:** …

## A2 · Eine eigene Lesernotiz mit Beleg speichern

Öffne [DOW-UAP-D080, Seite 5](../../data/raw/originals/DOW-UAP-D080.pdf). Formuliere, was die Akte über die Abbildungen auf den Seiten 5–8 sagt. Trenne dabei Deine Interpretation von der originalen Portalbeschreibung.

Ergänze ein Notizdokument mit folgender Struktur:

| Feld | Inhalt |
| --- | --- |
| `_id` | `"note:" + source["entry_id"]`; eine Notiz pro Eintrag und Arbeitsbereich |
| `entry_id` | Referenz auf den Katalogeintrag |
| `origin` | `"workshop_manual_review"` |
| `review.finding` | Kurze eigene Bezeichnung des Befunds |
| `review.statement` | Deine nachvollziehbare Aussage |
| `review.evidence.asset_id` | Referenz auf das gelesene PDF-Asset |
| `review.evidence.pdf_pages` | Liste der belegenden PDF-Seiten, z. B. `[5]` |

`review` und `evidence` sind verschachtelte Dictionaries. Die IDs referenzieren Katalog und Asset; sie enthalten keine kopierten Originaldateien.

In [ ]:
# TODO: Ersetze None durch Dein verschachteltes Notizdokument.
reading_note = None

In [ ]:
if reading_note is None:
    print("OFFEN: Lesernotiz in A2 ergänzen.")
else:
    validate_note(reading_note,source)
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        save_note(db,names,reading_note)
        save_note(db,names,reading_note)
        assert db[names["reading_notes"]].count_documents({"entry_id":source["entry_id"]}) == 1
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        saved_note = db[names["reading_notes"]].find_one({"_id":reading_note["_id"]})
    assert saved_note == reading_note
    completed.add("Lesernotiz")
    pprint(saved_note)

### Herkunft und referentielle Integrität

Warum bewahren wir Lesernotizen in einer separaten Collection auf? Was prüft der Anwendungscode vor dem Schreiben? Welche dieser Regeln würde MongoDB allein aufgrund einer gespeicherten `entry_id` automatisch erzwingen?

**Deine Begründung:** …

## B1 · Ein optionales Quellfeld finden

Finde alle Katalogeinträge, in deren `source_metadata` das originale Feld **PDF Pairing** vorhanden ist. Verwende einen Filter mit Punktnotation und `$exists`.

Die ursprünglichen Feldnamen enthalten Leerzeichen; ein Schlüssel wie `"source_metadata.PDF Pairing"` ist dennoch zulässig. Prüfe zusätzlich die Gegenmenge mit `$exists: False`.

`$exists: True` bedeutet vorhanden, auch wenn ein Wert `null` wäre. Im vorbereiteten `source_metadata` wurden leere Quellfelder weggelassen; diese konkrete Abfrage findet deshalb nicht leere Originalangaben. Ein Feld namens PDF Pairing kommt auch bei anderen Medienarten vor. [MongoDB: $exists](https://www.mongodb.com/docs/manual/reference/operator/query/exists/).

In [ ]:
# TODO: Filter für ein vorhandenes source_metadata.PDF Pairing.
pairing_filter = None

In [ ]:
if pairing_filter is None:
    print("OFFEN: Filter in B1 ergänzen.")
else:
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        pairing_entries = list(db[names["catalog"]].find(pairing_filter,
            {"_id":0,"entry_id":1,"source_key":1,"media_type":1,"source_metadata.PDF Pairing":1}).sort([("source_key",1),("entry_id",1)]))
        missing_pairing = db[names["catalog"]].count_documents({"source_metadata.PDF Pairing":{"$exists":False}})
    display(pd.DataFrame(pairing_entries).head(8))
    assert len(pairing_entries) == 111 and missing_pairing == 264
    completed.add("Optionales Quellfeld")
    print("Mit Feld:",len(pairing_entries),"Ohne Feld:",missing_pairing)

## B2 · Die gefilterten Einträge nach Medienart zählen

Ergänze eine Pipeline mit `$match`, `$group`, `$project` und `$sort`. Sie soll dieselben gefilterten Einträge aus B1 nach `media_type` zählen. Verwende die Ausgabefelder `media_type` und `catalog_entries`.

**Zähleinheit:** ein Katalogdokument. Die Anzahl der Token im Pairing-Feld, die Anzahl aufgelöster Verweise und die Zahl verknüpfter Dateien sind andere Kennzahlen. Für diese Frage ist kein `$unwind` erforderlich.

In [ ]:
# TODO: Aggregation mit pairing_filter; COUNT entspricht hier $sum: 1.
pipeline_media = None

In [ ]:
if pairing_filter is None or pipeline_media is None:
    print("OFFEN: Filter und Pipeline in B2 ergänzen.")
else:
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        media_counts = pd.DataFrame(list(db[names["catalog"]].aggregate(pipeline_media)))
    display(media_counts)
    actual = dict(zip(media_counts["media_type"],media_counts["catalog_entries"]))
    assert len(media_counts)==3 and actual == {"PDF":62,"VID":25,"IMG":24}
    assert media_counts["catalog_entries"].sum() == 111
    completed.add("Aggregation")

### Was darf die Redaktion daraus folgern?

Weshalb bedeuten fehlende PDF-Pairing-Metadaten nicht, dass eine Akte keinen fachlichen Bezug zu anderen Akten haben kann? Warum ist eine PDF-Akte nicht automatisch frei von Illustrationen? Beziehe Dich auf Deine Lesernotiz.

**Deine Einordnung:** …

## C1 · Vertiefung: Zwei Bedingungen an dasselbe Array-Element

Finde Einträge, bei denen **ein und dasselbe Asset** sowohl `locator_type="url"` als auch `format_hint="pdf"` besitzt. Ergänze `$elemMatch` und erkläre, warum getrennte Bedingungen auf `assets.locator_type` und `assets.format_hint` allgemein von verschiedenen Array-Elementen erfüllt werden könnten. [MongoDB: $elemMatch](https://www.mongodb.com/docs/manual/reference/operator/query/elemMatch/).

In [ ]:
# Optional: Beide Bedingungen müssen dasselbe Asset betreffen.
pdf_asset_filter = None

In [ ]:
if pdf_asset_filter is not None:
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        pdf_asset_entries = list(db[names["catalog"]].find(pdf_asset_filter,{"_id":1,"source_key":1}))
    expected_ids = {d["_id"] for d in inputs["catalog"]
                    if any(a["locator_type"]=="url" and a["format_hint"]=="pdf" for a in d["assets"])}
    assert {d["_id"] for d in pdf_asset_entries} == expected_ids
    print("Katalogeinträge mit PDF-URL-Asset:",len(pdf_asset_entries))
else:
    print("VERTIEFUNG: Array-Filter noch offen.")

## C2 · Vertiefung: Flexibles Schema mit Regeln

Lies den [vorbereiteten Validator](../../schemas/mongodb/reading_note_validator.json). Welche Typregel gilt für `pdf_pages`? Welche Zusatzfelder bleiben zulässig?

Die Probe verwendet eine eigene kurzlebige Collection. Sie prüft, ob MongoDB eine Seitenangabe als Text ablehnt, und zeigt anschliessend, dass eine wohlgeformte Referenz auf einen nicht vorhandenen Eintrag die Typprüfung trotzdem bestehen kann. Die eigentliche Notiz-Collection wird dabei nicht verändert.

Der Validator wird beim Erstellen der Probe-Collection gesetzt. Dadurch genügt auch das native Kurskonto mit `readWrite`; ein nachträgliches `collMod` ist nicht erforderlich. [MongoDB: Schemavalidierung](https://www.mongodb.com/docs/manual/core/schema-validation/).

In [ ]:
# Auf True setzen, wenn Du die optionale Server-Validierung prüfen möchtest.
RUN_SCHEMA_PROBE = False
note_validator = json.loads((ROOT / "schemas/mongodb/reading_note_validator.json").read_text(encoding="utf-8"))

In [ ]:
if RUN_SCHEMA_PROBE and reading_note is not None:
    with mongo_workspace(DATASET,IS_SOLUTION) as (db,names):
        schema_result = probe_note_schema(db,note_validator,reading_note)
    pprint(schema_result)
else:
    print("VERTIEFUNG: Schema-Probe nicht ausgeführt.")

## Abschluss

Die technischen Checks prüfen Struktur und Referenzzahlen. Besprecht zusätzlich die Einbettungsentscheidung und die inhaltliche Aussage mit Seitenbeleg. Ein erneuter Katalogimport erhält die separat gespeicherte Notiz.

In [ ]:
required = {"Import","Lesernotiz","Optionales Quellfeld","Aggregation"}
missing = sorted(required-completed)
if missing:
    print("AUFGABE OFFEN:",", ".join(missing))
else:
    print("MONGODB AUFGABE TECHNISCH OK. Modellentscheidung und Quelleninterpretation gemeinsam besprechen.")